**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion Models

The generative method behind modern image synthesis, told as a *signal processing* story: corrupt data with Gaussian noise step by step, train a network to **denoise**, then run the corruption in reverse. We train a complete diffusion model on 2-D data in minutes and watch noise crystallize into structure.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (Gaussians compose).
- [Representation Learning](./Representation_Learning.ipynb) S2 for the generative-model context.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# target distribution: two moons — structured, multimodal, low-dimensional
def moons(n):
    t = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t), np.sin(t)], 1)
    bot = np.stack([1 - np.cos(t), 0.4 - np.sin(t)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return ((X - X.mean(0)) / X.std(0)).astype(np.float32)

X = torch.from_numpy(moons(6000))
plt.figure(figsize=(3.6, 3.2)); plt.scatter(*X.T, s=2, alpha=0.3)
plt.title("the distribution we want to SAMPLE from"); plt.axis("equal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2694033/3980811452.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 1 of 2 — *The Forward Process & the Denoising Objective* (~35 min)
**Goal:** destroy data with scheduled noise; train a network to predict the noise.
**Feeds into:** Session 2 (sampling = reverse diffusion).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Forward Process & the Denoising Objective</b></summary>

**Timing (~35 min).** 10 min why generation is hard and denoising is easy · 10 min the forward process and the closed form · 10 min the loss · 5 min the time embedding.

**Open with the reframing that is the entire idea.** Sampling from a complicated distribution is hard. **Removing a little noise is easy** — it is a regression problem, the cousin of Wiener filtering in [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb). Diffusion's insight is to *chain the easy problem*: define a process that destroys data gradually, learn to undo one step, then run the chain backwards. Nobody ever asks the network to generate anything.

**Derive the closed form rather than quoting it, because it is what makes training tractable.** Each forward step adds a little Gaussian noise; Gaussians compose, so the composition of $t$ steps is itself Gaussian and you get $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ in **one formula**. That is why the training loop can sample a random $t$ and jump straight there — no simulation of the chain, no sequential rollout. Without this identity, training would cost $O(T)$ per example and the method would be impractical.

**Then the loss, and let its plainness land.** `((model(xt, t) - eps)**2).mean()`. That is the whole objective: **predict the noise you added**. No adversary, no ELBO, no reconstruction term, no KL. Students who have seen GAN training instability or VAE posterior collapse find this genuinely surprising, and the stability of diffusion training is most of why it displaced both. The variational derivation exists and reduces to this after simplification — worth mentioning, not worth deriving here.

**Explain the time embedding in one sentence, since it looks like a magic trick.** One network handles all 200 noise levels, so it must be *told* which one it is facing. Sinusoidal features of $t$ are the same positional encoding used in transformers: a smooth, high-frequency-rich representation that lets the network condition sharply on $t$ without needing 200 separate models.

**Prepare the room to read the loss values correctly, because they look wrong.** They go 0.92, 0.32, 0.41, 0.35 — **non-monotone**, and that is expected. Each print is a single 256-sample minibatch with a random $t$, and the achievable loss depends strongly on $t$. At large $t$, $x_t \approx \varepsilon$, so predicting $\varepsilon$ is nearly free and the loss is small. At small $t$, $x_t \approx x_0$ and the noise has almost no visible effect, so $\varepsilon$ is close to unrecoverable and the loss approaches 1 (the variance of $\varepsilon$). **The bouncing is $t$-sampling variance, not instability**, and a room that is not warned will conclude training failed.

**Give the DSP reading of the forward pictures, since this audience owns it.** Noising is progressive low-pass-plus-noise: fine detail drowns first, coarse structure survives longest. That is why the reverse process builds gross shape early and detail late — **generation as spectral refinement** — and why the same model does denoising, inpainting, and super-resolution as *partial* trips along the chain.

**One honest detail about the schedule worth checking with the room.** With $\beta$ from $10^{-4}$ to $0.04$ over 200 steps, $\bar\alpha_T \approx 0.017$, so $\sqrt{\bar\alpha_T} \approx 0.13$ — the final "pure noise" still carries about **13% of the original signal amplitude**. Sampling starts from exact $\mathcal{N}(0, I)$, so there is a small mismatch between where the forward process ends and where the reverse process begins. It is minor here and it is the reason production schedules use more steps or a cosine profile.
</details>

## 2. Destruction Is Easy — Learn to Undo It

💡 **Intuition.** Generating from scratch is hard; *removing a little noise* is easy — it's [Wiener denoising's](../Intro_DSP/Statistical_Signal_Processing.ipynb) cousin, a regression problem. Diffusion's insight: chain the easy problem. Define a forward process that gradually noises data into pure Gaussian ($x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ — Gaussians compose, so any step is one formula), and train one network $\varepsilon_\theta(x_t, t)$ to **predict the noise** that was added. Denoising at *every* noise level = knowing the path from chaos back to data.

In [2]:
T = 200
betas = torch.linspace(1e-4, 0.04, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                      # ᾱ_t

# visualize the forward death of the data
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for ax, t_show in zip(axes, [0, 60, 120, 199]):
    eps = torch.randn_like(X)
    xt = abar[t_show].sqrt()*X + (1-abar[t_show]).sqrt()*eps
    ax.scatter(*xt.T, s=1, alpha=0.2); ax.set_title(f"t={t_show}"); ax.axis("equal"); ax.set_xlim(-3,3); ax.set_ylim(-3,3)
plt.suptitle("forward process: structure dissolves into N(0, I)")
plt.tight_layout(); plt.show()

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


/tmp/ipykernel_2694033/3661612610.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Four snapshots of the same 6,000 points being destroyed. At $t = 0$ the two moons are crisp. By $t = 60$ they are smeared but still recognisably two arcs. At $t = 120$ only a vague elongation survives. At $t = 199$ it is a featureless Gaussian blob — the structure is gone.

**Read the order of destruction, because it is the whole DSP story.** Fine detail dies first; **coarse structure survives longest**. The gap between the moons persists long after the crispness of each arc has dissolved. That is progressive low-pass-plus-noise: adding white noise raises the floor uniformly, so the low-amplitude high-frequency content disappears under it before the large-scale shape does.

**Which immediately tells you what the reverse process must do.** If detail dies last-in, it is recovered first-out — the reverse chain builds **gross shape early and detail late**. Generation is spectral refinement. And this explains, in one sentence, why the same trained model can denoise, inpaint, and super-resolve: those tasks are *partial* trips along a chain the model already knows end to end.

**Note that no simulation was needed to draw this figure.** Each panel jumps straight to its $t$ via $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\varepsilon$ — one line, because Gaussians compose and a chain of Gaussian steps is a single Gaussian step with accumulated variance. **That closed form is what makes training affordable**: the loop samples a random $t$ per example and jumps there, instead of rolling the chain forward $t$ times. Without it the method costs $O(T)$ per training example and nobody builds it.

**Check the final panel against the schedule, though, because "pure noise" is an approximation here.** With $\beta$ running from $10^{-4}$ to $0.04$ over 200 steps, $\sum\beta_t \approx 4.0$ and $\bar\alpha_T \approx 0.017$ — so $\sqrt{\bar\alpha_T} \approx 0.13$ and about **13% of the original signal amplitude is still present** at $t = 199$. Sampling in Session 2 starts from exact $\mathcal{N}(0, I)$, so the reverse chain begins at a slightly different distribution from where the forward chain ended.

**The mismatch is small and worth knowing about rather than worrying about.** It costs a little sample quality and is invisible at this scale, but it is exactly why production schedules use 1000 steps, or a cosine $\bar\alpha$ profile that drives the signal much closer to zero at the end. **A demonstration that admits its own approximations is more useful than one that asserts "now it is pure noise"** — and you can verify this one directly by printing `abar[-1].sqrt()`.

In [3]:
# the denoiser: predicts ε from (x_t, t) — t is embedded sinusoidally, like a transformer position
class Denoiser(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU(),
                                 nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 2))
    def t_embed(self, t):
        k = torch.arange(8)
        ang = t[:, None] / T * (100 ** (k / 8))[None]
        return torch.cat([ang.sin(), ang.cos()], 1)
    def forward(self, x, t):
        return self.net(torch.cat([x, self.t_embed(t.float())], 1))

model = Denoiser()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(4000):
    ix = torch.randint(0, len(X), (256,))
    x0 = X[ix]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = abar[t, None].sqrt()*x0 + (1-abar[t, None]).sqrt()*eps
    loss = ((model(xt, t) - eps)**2).mean()            # predict the noise. that's the whole loss.
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0: print(f"step {step:4d}  denoising loss {loss.item():.4f}")

step    0  denoising loss 0.9189


step 1000  denoising loss 0.3236


step 2000  denoising loss 0.4127


step 3000  denoising loss 0.3509


**What just happened.** A complete diffusion model trained in 4,000 steps on a laptop, and the loss reads **0.9189 → 0.3236 → 0.4127 → 0.3509**.

**Those numbers are not monotone, and that is expected rather than a failure.** Each print is a *single* 256-sample minibatch with a randomly drawn $t$, and the achievable loss depends heavily on which $t$ was drawn. At large $t$, $x_t \approx \varepsilon$, so the network can nearly read the answer off its input and the loss is small. At small $t$, $x_t \approx x_0$ and the noise has barely perturbed anything, so $\varepsilon$ is close to unrecoverable and the loss approaches **1.0** — the variance of $\varepsilon$ itself. **The bouncing is $t$-sampling variance, not instability.** A room not warned about this concludes the run diverged.

**Which gives you the right yardstick: 1.0, not 0.** A network that outputs zero achieves MSE 1.0, since $\varepsilon \sim \mathcal{N}(0,I)$. So 0.35 means the model explains about 65% of the noise variance averaged over $t$ — and no model can drive this to zero, because at $t$ near 0 the noise genuinely is not identifiable from the input. **The loss floor is a property of the problem, not of the network.**

**Now look at the objective itself, because its plainness is the point of the workshop.** `((model(xt, t) - eps)**2).mean()` — predict the noise you added. No adversary, no ELBO, no reconstruction term, no KL, no posterior. Anyone who has fought GAN mode collapse or VAE posterior collapse should find this startling: **the training stability of diffusion is most of why it displaced both**. The variational derivation exists and simplifies down to exactly this expression.

**Two implementation details are doing real work.** The **time embedding** — sinusoidal features of $t$, the same encoding transformers use for position — is what lets *one* network serve all 200 noise levels; without it you would need 200 models or a model that cannot tell which regime it is in. And **SiLU** rather than ReLU: the denoiser must output a smooth vector field, and ReLU's kink produces visible artifacts in the sampling trajectory.

**Finally, note the scale of what this is.** 4,000 steps, a 4-layer MLP, two dimensions, under a minute. Stable Diffusion is **the same loss and the same schedule** with a U-Net denoiser, a latent space, text conditioning, and six orders of magnitude more compute. The idea does not get more complicated as it gets bigger — which is precisely why it is worth training one at this scale first.

---
### 🕐 Session 2 of 2 — *Sampling: Running Time Backwards* (~40 min)
**Goal:** start from pure noise and iteratively denoise into fresh samples.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Sampling — Running Time Backwards</b></summary>

**Timing (~40 min).** 10 min the reverse step · 12 min running it and reading the snapshots · 12 min the quantitative audit · 6 min what the audit cannot see.

**Walk the sampling loop line by line, because every line is a decision.** Start at $x_T \sim \mathcal{N}(0,I)$. Ask the network for $\hat\varepsilon$. **Invert the forward formula** to get $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon)/\sqrt{\bar\alpha_t}$ — a guess at the clean data. Then re-noise that guess to level $t-1$ and add a little fresh noise. Repeat 200 times. The key insight to state aloud: **the model never generates; it repeatedly guesses "what would this have been without noise?" and takes one step in that direction.**

**Explain the fresh-noise injection, since students always ask why you would add noise back.** Without it the map is deterministic and every starting point flows to one destination — you get DDIM, which is fine and faster but produces less diverse samples per unit of exploration. The stochasticity is what lets nearby starting points land in genuinely different places, and it is also what makes the chain a proper reverse-time SDE rather than an ODE. Both are legitimate; they are different samplers for the same trained model.

**Have the room predict the snapshots.** Given that the forward process destroyed detail before structure, what should come back first? **Gross shape early, detail late** — and the panels show exactly that: a blob, then two lobes, then two arcs, then crisp moons. This is the spectral-refinement story from Session 1 running in reverse, and it is worth calling out as a *prediction confirmed*, not just a nice picture.

**Then insist on the audit, because a pretty scatter plot is not evidence.** Means match to 0.04; covariances match to about 5%; the 90th-percentile nearest-neighbour distance to the real data is **0.040** on unit-variance data. Three independent checks, all passing. That discipline — do not accept "it looks right" — is the transferable habit of this session.

**But be rigorous about what the audit cannot detect, and give this real time.** Nearest-neighbour distance is **minimised by memorisation**: a model that reproduced training points exactly would score 0.000 and look perfect by this metric. It also cannot see **mode collapse** — generate only the upper moon and the distance stays small while the covariance shifts only moderately. Ask the room how to detect these two failures before telling them. The answers are a *reverse* nearest-neighbour check (how far is each training point from its nearest generated point?) and per-mode counts.

**Two-moons is 2-D, which is what makes any of this checkable at all.** In image space, "does the generated distribution match the data distribution?" has no reliable answer — FID is a crude Gaussian proxy in a feature space, and every serious paper reports human evaluation because the metrics are known to be inadequate. **Low dimension is why this notebook can grade itself**, and the honest framing is that the audit here is far stronger than anything available at image scale.

**Close by scaling the idea up in one sentence.** Same loss, same schedule, same loop — swap the MLP for a U-Net, work in a latent space, add text conditioning, multiply the compute by $10^6$, and you have Stable Diffusion. Nothing conceptual is added on the way. Then point at [Optimal Transport](./Optimal_Transport.ipynb): the probability-flow view of this reverse path is a transport map, and flow matching trains it with straighter, OT-inspired couplings so that fewer integration steps are needed at sampling time.
</details>

## 3. The Reverse Process

💡 **Intuition.** To sample: start at $x_T \sim \mathcal{N}(0, I)$ and repeatedly apply the learned denoiser, stepping $t = T{-}1, \dots, 0$, re-injecting a *little* fresh noise each step (the stochasticity keeps samples diverse — drop it and you get DDIM's deterministic cousin). Each step is a small, easy denoise; a few hundred of them compound into creation. Image generators are this exact loop with a U-Net denoiser and billions of pixels.

In [4]:
@torch.no_grad()
def sample(n_samp=3000, snapshots=(199, 120, 60, 0)):
    x = torch.randn(n_samp, 2)
    shots = {}
    for t_i in reversed(range(T)):
        t = torch.full((n_samp,), t_i)
        eps_hat = model(x, t)
        x0_hat = (x - (1-abar[t_i]).sqrt()*eps_hat) / abar[t_i].sqrt()
        if t_i > 0:
            ab_prev = abar[t_i-1]
            x = ab_prev.sqrt()*x0_hat + (1-ab_prev-betas[t_i]).clamp(min=0).sqrt()*eps_hat                 + betas[t_i].sqrt()*torch.randn_like(x)
        else:
            x = x0_hat
        if t_i in snapshots: shots[t_i] = x.clone()
    return shots

shots = sample()
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for ax, (t_i, xs) in zip(axes, sorted(shots.items(), reverse=True)):
    ax.scatter(*xs.T, s=1, alpha=0.2); ax.set_title(f"t={t_i}")
    ax.axis("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle("reverse process: noise crystallizes into the two moons")
plt.tight_layout(); plt.show()

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


/tmp/ipykernel_2694033/2917646786.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three thousand points of pure Gaussian noise walked backwards through 200 denoising steps and **became two moons**. The snapshots run blob → elongated cloud → two lobes → crisp arcs. Nothing in the model was ever asked to generate anything; it was asked, 200 times, "what would this have been without the noise?"

**Compare the order of appearance against the forward process, because it is a prediction being confirmed.** Session 1 showed detail dying first and coarse structure surviving longest. Run the chain backwards and the reverse must hold: **gross shape first, detail last**. The panels show exactly that — the gap between the moons emerges long before either arc becomes sharp. Generation as spectral refinement, low frequencies to high, and the figure is evidence for it rather than an illustration of it.

**The key line is the inversion, and it is worth reading aloud.** `x0_hat = (x - (1-abar).sqrt()*eps_hat) / abar.sqrt()` is the forward formula solved for $x_0$. The network predicts the noise; algebra converts that into a guess at the clean data; the guess is then re-noised to level $t-1$. **At no point does the model output a sample** — it outputs a noise estimate, and the sampler does the rest.

**The fresh noise added at each step is a deliberate choice with a named alternative.** `betas[t_i].sqrt()*torch.randn_like(x)` keeps the chain stochastic, so nearby starting points can land in genuinely different places. Delete it and the map becomes deterministic — that is DDIM, which is faster and allows far fewer steps, at the cost of less diversity per unit of exploration. Same trained model, different sampler; both are legitimate and the choice is made at sampling time.

**Note the cost asymmetry, since it is the method's defining practical weakness.** Training touched one random $t$ per example. Sampling requires **200 sequential forward passes per sample**, and they cannot be parallelised across $t$ because each depends on the last. That is why diffusion image generation takes seconds where a GAN takes milliseconds, and why an entire literature — DDIM, DPM-Solver, consistency models, distillation — exists purely to cut the step count. The 200 here could probably be 20 with a better sampler.

**One caveat on where the chain starts.** Sampling begins at exact $\mathcal{N}(0, I)$, but the forward process at $t = 199$ left about 13% of the signal amplitude in place ($\sqrt{\bar\alpha_T} \approx 0.13$). So the reverse chain starts from a slightly different distribution than the one the forward chain reached — a small train/sample mismatch, invisible at this scale, and the reason real schedules use more steps or a cosine profile.

**And resist declaring victory from the picture alone.** It looks right, which is not the same as being right — the moons could be subtly the wrong width, one mode could be underpopulated, or the model could be reproducing training points. The next cell exists precisely because "it looks like the data" is not a measurement.

In [5]:
# quantitative check: do generated samples match the data's statistics?
gen = shots[0]
def stats(A):
    A = np.asarray(A)
    return A.mean(0).round(2), np.cov(A.T).round(2)
m_d, C_d = stats(X); m_g, C_g = stats(gen)
print("data  mean", m_d, " cov\n", C_d)
print("model mean", m_g, " cov\n", C_g)
# and the harder test: fraction of generated points close to the true manifold
from scipy.spatial import cKDTree
dist, _ = cKDTree(X.numpy()).query(gen.numpy())
print(f"\n90th-percentile distance of generated points to the data manifold: {np.quantile(dist, 0.9):.3f}")

data  mean [ 0. -0.]  cov
 [[ 1.   -0.47]
 [-0.47  1.  ]]
model mean [0.04 0.02]  cov
 [[ 1.06 -0.52]
 [-0.52  1.02]]

90th-percentile distance of generated points to the data manifold: 0.040


**What just happened.** Three independent checks on whether the generated distribution actually matches the data, rather than merely looking like it:

| statistic | data | generated |
|---|---|---|
| mean | $[0.00, -0.00]$ | $[0.04, 0.02]$ |
| variances | $1.00,\ 1.00$ | $1.06,\ 1.02$ |
| correlation | $-0.47$ | $-0.52$ |

and the 90th-percentile nearest-neighbour distance from a generated point to the real data is **0.040** — on unit-variance data, so 4% of a standard deviation. Nine tenths of the generated points sit essentially *on* the manifold.

**The systematic 2–6% variance overshoot is worth noticing rather than rounding away.** Both variances came out above 1.0 and the correlation is slightly stronger than the data's. That is the expected signature of a diffusion sampler that has not fully converged: residual noise from the last few steps has not been removed, so samples are slightly more spread than the target. It is small, it is consistent across both coordinates, and it would shrink with more training steps or a finer schedule.

**Now the important part: what these numbers cannot see.** The nearest-neighbour distance is **minimised by memorisation**. A model that simply reproduced training points would score **0.000** and look perfect by this metric. So 0.040 is evidence the samples are *on* the manifold and no evidence at all that they are *new*. For a generative model, the second question is the one that matters — and this audit does not ask it.

**It is also blind to mode collapse.** Generate only the upper moon and the nearest-neighbour distance stays tiny, while the mean shifts a little and the covariance changes moderately — plausibly within what is reported above. **All three statistics can pass while half the distribution is missing.**

**Both gaps have cheap fixes, and they are the experiment worth running.** For coverage, compute the **reverse** nearest-neighbour distance: for each *training* point, how far is the closest *generated* point? If a mode is missing, that number explodes for the abandoned region while the forward direction stays small. For novelty, compare the forward distance against the typical nearest-neighbour distance *within* the training set — if generated points are systematically closer to training data than training data is to itself, the model is copying.

**Two-moons is 2-D, which is exactly why any of this is checkable.** In image space there is no reliable answer to "does the generated distribution match the data distribution?" — FID is a Gaussian approximation in a feature space chosen for other purposes, and every serious paper still reports human evaluation because the metrics are known to be inadequate. **The audit available here is far stronger than anything available at image scale**, and that is worth knowing before trusting a headline number from a generative-model paper.

**Still, the honest summary is a good one.** One regression loss, one noise schedule, 4,000 training steps on a laptop, and the generated distribution matches the target's first and second moments to a few percent with 90% of samples on the manifold. Swap the MLP for a U-Net and add six orders of magnitude of compute and this is Stable Diffusion — the idea does not get more complicated on the way up.

**The DSP lens, explicitly:** the forward process is progressive low-pass-plus-noise (coarse structure survives longest); the reverse process therefore builds coarse structure first and details last — generation as *spectral refinement*. That's also why diffusion models are natural denoisers, inpainters, and super-resolvers: those are just partial trips along the same chain.

## 4. Conclusion

One regression loss (predict the noise), one schedule, and a walk backwards through it: that's the entire method behind modern generative imagery — and you just trained one.

---
## Where next

- [Representation Learning](./Representation_Learning.ipynb) — VAEs: the previous generation of generation.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — the denoising theory underneath.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — what it takes to run this at image scale.